# Real-data scatterplots

All training observations from split 01 (blue) and the same number of GEB posterior predictive samples (red). GEB is fitted using all analysis coordinates of that training split. POWER uses the mixture transform; Dry Bean and ISOLET use the Gaussian transform. Both rows show the same coordinate pairs on the transformed scale.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

root = Path.cwd().resolve()
if root.name == 'notebooks':
    root = root.parent

if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from experiment_utils.transforms import (
    apply_mixture_cdf, apply_normal_cdf, fit_mixture_cdf, fit_normal_cdf,
)

from methods.GEB import GEBTree

panels = [
    ('household_power', 'POWER', 'Voltage', 'Sub_metering_3', 'Voltage', 'Sub-metering 3', fit_mixture_cdf, apply_mixture_cdf),
    ('dry_bean', 'Dry Bean', 'Perimeter', 'AspectRatio', 'Perimeter', 'Aspect ratio', fit_normal_cdf, apply_normal_cdf),
    ('isolet', 'ISOLET', 'PC1', 'PC100', 'PC1', 'PC100', fit_normal_cdf, apply_normal_cdf),
]

fig, axes = plt.subplots(2, 3, figsize=(10, 6.4), layout='constrained')
for column, (dataset, title, x_feature, y_feature, xlabel, ylabel, fit, apply) in enumerate(panels):
    path = root / 'data' / 'train_test' / dataset / 'repetition_01_train.npz'
    with np.load(path, allow_pickle=False) as split:
        names = split['feature_names'].tolist()
        train = split['x']
    coordinates = [names.index(x_feature), names.index(y_feature)]
    train = apply(train, fit(train))
    n = len(train)

    d = train.shape[1]
    model = GEBTree(d, np.zeros(d), np.ones(d), alpha=1., dataset=train, m=32)
    sampled = model.tree_sampling(n, random_state=2026)
    for row, (points, color, label) in enumerate([
        (train, '#4C78A8', 'Training data'),
        (sampled, '#D95F5F', 'GEB predictive samples'),
    ]):
        x, y = points[:, coordinates].T
        ax = axes[row, column]
        ax.scatter(x, y, s=5, color=color, alpha=0.3, linewidths=0, rasterized=True)
        ax.tick_params(axis='x', labelbottom=row == 1)
        ax.set(title=f'{title} — {label}', xlabel=xlabel if row == 1 else '', ylabel=ylabel,
               xlim=(0, 1), ylim=(0, 1))

output = root / 'results' / 'real_data_scatterplots'
output.mkdir(parents=True, exist_ok=True)
fig.savefig(output / 'real_data_scatterplots.pdf', dpi=250)
plt.show()
